# Distributed Training

> The previous notebook computed that training a 7B model with AdamW needs roughly 112 GB of fixed memory (parameters + gradients + optimizer state). A single A100 (80 GB) cannot hold this, let alone leave room for activations. Compute scales out, but per-GPU memory hits a physical ceiling.
>
> This notebook answers how multiple GPUs share the work. We start from plain Data Parallel (DDP) to see exactly which memory on each card is redundant copy. ZeRO's three Stages then carve that redundancy across cards, one category at a time. We finish by comparing FSDP and DeepSpeed — two production implementations of ZeRO — and Accelerate, a thin dispatcher on top.

There are three ways to split training across GPUs. Data Parallel keeps a full model replica on each card and splits only the data. Model Parallel partitions the model itself along some dimension, with each card computing a slice. Pipeline Parallel cuts the model by layers and pipelines different micro-batches through the stages. The three patterns suit different situations, and large pre-training runs usually mix them.

This notebook focuses on the data-parallel lineage. Two reasons: first, it is the most common and most easily understood pattern; second, the mainstream industrial solutions — ZeRO, FSDP, DeepSpeed — are all variations on data parallelism that shard the memory more finely. Once you see where DDP's redundancy lives and what ZeRO's three Stages do, the differences between the implementations become obvious.


## 1. The single-GPU ceiling: re-checking the memory budget

Let us put the memory budget on the table. Under AdamW with mixed precision, each parameter occupies roughly 16 bytes of memory: FP16 parameters (2 bytes), FP16 gradients (2 bytes), FP32 master weights (4 bytes), AdamW first moment m (4 bytes), AdamW second moment v (4 bytes). This is the fixed overhead independent of training step count; activations come on top.


In [ ]:
# === Single-GPU memory budget: 7B model ===
P = 7e9  # 7B parameters

param_fp16   = 2 * P   # FP16 parameters
grad_fp16    = 2 * P   # FP16 gradients
master_fp32  = 4 * P   # FP32 master weights
adam_m       = 4 * P   # AdamW first moment
adam_v       = 4 * P   # AdamW second moment

fixed_bytes = param_fp16 + grad_fp16 + master_fp32 + adam_m + adam_v
fixed_gb = fixed_bytes / 1e9

print(f"Parameters (FP16):   {param_fp16 / 1e9:.1f} GB")
print(f"Gradients (FP16):    {grad_fp16 / 1e9:.1f} GB")
print(f"Master weights (FP32): {master_fp32 / 1e9:.1f} GB")
print(f"AdamW m (FP32):      {adam_m / 1e9:.1f} GB")
print(f"AdamW v (FP32):      {adam_v / 1e9:.1f} GB")
print(f"Fixed overhead:      {fixed_gb:.0f} GB (= 16 × P bytes)")
print()
print(f"A single A100 (80 GB) cannot fit {fixed_gb:.0f} GB of fixed overhead, plus activations.")


## 2. Data Parallel (DDP)

The simplest idea is Data Parallel (DDP): each card holds a full model replica, the dataset is split into N shards across N cards, and each card independently runs forward and backward. After backward, an all-reduce averages gradients across cards. Every card sees a different batch but updates the same model — effectively training with N× the batch size.

DDP's headline benefit is linear throughput scaling: 8 cards approach 8× single-card throughput. The cost is no memory saving — every card still stores the full 112 GB of fixed overhead. Total memory across 8 cards is 8 × 112 = 896 GB, of which 7/8 is redundant: the same optimizer states, the same master weights, replicated on every card.


In [ ]:
# === DDP memory cost ===
N = 8  # 8 cards
P = 7e9

per_card_bytes = 16 * P   # DDP: every card has a full replica
total_bytes = N * per_card_bytes

print(f"DDP setup: {N} cards, each = {per_card_bytes / 1e9:.0f} GB fixed overhead")
print(f"Total memory = {N} × {per_card_bytes / 1e9:.0f} = {total_bytes / 1e9:.0f} GB")
print(f"Redundant portion = {total_bytes / 1e9 - per_card_bytes / 1e9:.0f} GB ({(N-1)/N*100:.0f}% is replica)")
print()
print("Key observation: DDP achieves linear throughput growth but saves zero memory.")
print("If we could shard those redundant replicas across cards, we could train larger models.")


## 3. ZeRO: sharding the redundancy

ZeRO (Zero Redundancy Optimizer) is a method proposed by Microsoft in 2020. Its starting point is straightforward: under DDP, the per-card fixed overhead of 16P bytes splits into three parts — optimizer state (12P bytes, including master weights, m, v), gradients (2P bytes), and parameters (2P bytes). All three are redundant copies and can each be sharded across cards.

ZeRO decomposes the sharding process into three progressive Stages, each adding one more category. Stage 1 shards optimizer state, Stage 2 adds gradients, Stage 3 adds parameters. Below we hand-calculate how much memory each Stage saves on 8 cards.


### 3.1 Stage 1: shard optimizer state

AdamW's optimizer state (master weights, m, v) totals 12P bytes. Stage 1 shards this 12P across N cards, so each card holds only 12P/N. Gradients (2P) and parameters (2P) are still fully replicated on every card.

After sharding, per-card memory = 2P (parameters) + 2P (gradients) + 12P/N (optimizer state).


In [ ]:
# === ZeRO Stage 1 hand calculation ===
P = 7e9
N = 8

s1_optimizer = 12 * P / N
s1_param = 2 * P
s1_grad = 2 * P
s1_total = s1_optimizer + s1_param + s1_grad

print(f"Stage 1 per-card memory (7B × {N} cards):")
print(f"  Parameters (FP16, full):     {s1_param / 1e9:.1f} GB")
print(f"  Gradients (FP16, full):      {s1_grad / 1e9:.1f} GB")
print(f"  Optimizer state (1/{N} shard): {s1_optimizer / 1e9:.1f} GB")
print(f"  Per-card subtotal:           {s1_total / 1e9:.1f} GB")
print()
print(f"Compared to DDP's 112 GB, Stage 1 brings it down to {s1_total / 1e9:.1f} GB.")
print("Cost: after backward, gradients are reduce-scattered so each card keeps only the slice matching its optimizer shard.")


### 3.2 Stage 2: also shard gradients

Stage 1 leaves a symmetry break: gradients are only 2P bytes but every card still keeps a full copy. Stage 2 also shards gradients by N cards, so each card keeps only the slice corresponding to its optimizer shard.

After sharding, per-card memory = 2P (parameters) + 2P/N (gradients) + 12P/N (optimizer state).


In [ ]:
# === ZeRO Stage 2 hand calculation ===
P = 7e9
N = 8

s2_optimizer = 12 * P / N
s2_grad = 2 * P / N
s2_param = 2 * P
s2_total = s2_optimizer + s2_grad + s2_param

print(f"Stage 2 per-card memory (7B × {N} cards):")
print(f"  Parameters (FP16, full):     {s2_param / 1e9:.1f} GB")
print(f"  Gradients (1/{N} shard):      {s2_grad / 1e9:.2f} GB")
print(f"  Optimizer state (1/{N} shard): {s2_optimizer / 1e9:.1f} GB")
print(f"  Per-card subtotal:           {s2_total / 1e9:.1f} GB")


### 3.3 Stage 3: also shard parameters

Stage 3 shards the last piece of redundancy — the parameters themselves. Each card holds only 1/N of the parameters. When forward and backward need the parameters, they all-gather the full layer on the fly, then discard it after use.

After sharding, per-card memory = 16P/N (all three categories sharded). This is ZeRO's limit.


In [ ]:
# === ZeRO Stage 3 hand calculation ===
P = 7e9
N = 8

s3_total = 16 * P / N

print(f"Stage 3 per-card memory (7B × {N} cards):")
print(f"  Parameters + gradients + optimizer state (all 1/{N} shard): {s3_total / 1e9:.1f} GB")
print()
print(f"Compared to DDP's 112 GB, Stage 3 brings it down to {s3_total / 1e9:.1f} GB.")
print("Cost: forward and backward must all-gather the current layer's full parameters every time, significantly increasing communication.")


### Three Stages side by side

Putting all three Stages together for an 8-card, 7B setup:


In [ ]:
# === ZeRO three-Stage comparison ===
P = 7e9

print(f"{'Method':<22} {'Per-card':>12} {'vs DDP':>12}")
print("-" * 50)
configs = [
    ("DDP",           16 * P),
    ("ZeRO Stage 1",  2 * P + 2 * P + 12 * P / 8),
    ("ZeRO Stage 2",  2 * P + 2 * P / 8 + 12 * P / 8),
    ("ZeRO Stage 3",  16 * P / 8),
]
ddp_bytes = 16 * P
for name, total in configs:
    ratio = total / ddp_bytes
    print(f"{name:<22} {total / 1e9:>10.1f} GB   {ratio * 100:>6.1f}%")
print()
print("From DDP to Stage 3, per-card memory compresses to 1/8. Communication cost also rises with each Stage.")


## 4. FSDP: PyTorch-native ZeRO-3

ZeRO is an algorithm from a paper; in production there are two main implementations: FSDP and DeepSpeed. FSDP first.

FSDP (Fully Sharded Data Parallel) is a module built into PyTorch since 1.11, positioned as the official implementation of ZeRO Stage 3. It shards parameters, gradients, and optimizer state across cards, and all-gathers full parameters on demand during forward and backward. The API is very close to DDP, so migration cost is mainly the wrapping line:


In [ ]:
# === Minimal DDP vs FSDP code diff (pseudo-code, cannot run in single process) ===
# This snippet is a structural comparison only — we do not actually launch multi-process here.

ddp_code = '''
from torch.nn.parallel import DistributedDataParallel as DDP

model = MyModel().cuda()
model = DDP(model)              # ← key difference
optimizer = torch.optim.AdamW(model.parameters())
'''

fsdp_code = '''
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

model = MyModel().cuda()
model = FSDP(model)             # ← key difference
optimizer = torch.optim.AdamW(model.parameters())
'''

print("DDP version:")
print(ddp_code)
print("FSDP version:")
print(fsdp_code)
print("Difference: swap DDP for FSDP; the rest of the training loop stays the same.")
print("FSDP internally handles all-gather / reduce-scatter; the exposed interface matches DDP.")


FSDP's advantage is being bundled with PyTorch mainline, so it tracks NCCL and CUDA new features fastest. Its downside is a larger configuration surface — MixedPrecision, ShardingStrategy, CPUOffload all need explicit settings. It is the first choice when fine control matters; when you just want to get running quickly, DeepSpeed's config-driven approach is more convenient.


## 5. DeepSpeed: config-driven ZeRO

DeepSpeed is Microsoft's open-source training framework. It exposes Stage selection, offload strategy, and communication optimizations as JSON config fields. Training code barely changes — you pass `--deepspeed_config` at launch.

Below is a minimal config enabling ZeRO Stage 2:


In [ ]:
# === DeepSpeed minimal config example ===
import json

deepspeed_config = {
    "train_micro_batch_size_per_gpu": 4,
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {
            "device": "none"          # do not offload to CPU, keep GPU speed
        }
    },
    "fp16": {
        "enabled": True
    }
}

print("deepspeed_config.json:")
print(json.dumps(deepspeed_config, indent=2, ensure_ascii=False))
print()
print("Launch command:")
print("  deepspeed --num_gpus=8 train.py --deepspeed_config deepspeed_config.json")
print()
print("Changing stage to 3 enables ZeRO-3 (equivalent to FSDP).")
print('Changing offload_optimizer.device to "cpu" spills optimizer state to host RAM — saves more GPU memory but is slower.')


DeepSpeed offers two capabilities beyond FSDP: first, ZeRO-Offload, which can offload optimizer state and even parameters to CPU RAM or NVMe, buying more effective memory; second, ZeRO-Infinity, which pushes offload to the extreme and can train trillion-parameter models on 32 GPUs. The cost is the CPU↔GPU transfer overhead, slower than pure-GPU FSDP. Use only when the model is so large that GPU memory genuinely runs out.


## 6. Accelerate: a unified wrapper

Writing training scripts reveals that DDP, FSDP, and DeepSpeed each have different launch conventions: DDP uses `torchrun`, FSDP also uses `torchrun` but requires wrapping changes, DeepSpeed uses the `deepspeed` launcher. Switching backends means rewriting code, which is annoying.

Accelerate is HuggingFace's thin wrapper that hides these differences behind a single `Accelerator` object. The same training code can switch between DDP / FSDP / DeepSpeed just by changing the `accelerate config`:


In [ ]:
# === Accelerate's unified training loop (pseudo-code) ===
accelerate_code = '''
from accelerate import Accelerator

accelerator = Accelerator()

model, optimizer, dataloader, scheduler = accelerator.prepare(
    model, optimizer, dataloader, scheduler
)

for batch in dataloader:
    loss = model(batch)
    accelerator.backward(loss)        # ← replaces loss.backward()
    optimizer.step()
    optimizer.zero_grad()
'''

print("Training code:")
print(accelerate_code)
print()
print("Switching backend is just changing accelerate config, without touching code:")
print("  accelerate config  → interactively pick DDP / FSDP / DeepSpeed")
print("  accelerate launch train.py  → launch with the configured backend")


Accelerate is not another distributed algorithm — it is a dispatch layer. Under DDP mode it is torch.distributed, under FSDP mode it is PyTorch FSDP, under DeepSpeed mode it is DeepSpeed. Its value is "one codebase, multiple backends", well-suited for running experiments or writing cross-framework open-source projects.


## 7. Three layers in summary

Putting the three solutions side by side, they sit at different abstraction levels:


In [ ]:
# === Three-layer comparison table ===
rows = [
    ("Method", "Abstraction", "Sharding granularity", "Typical use"),
    ("PyTorch DDP", "lowest", "no sharding (data parallel)", "fits on single card"),
    ("PyTorch FSDP", "low (PyTorch built-in)", "ZeRO-3, per-layer wrapping", "fine-grained control"),
    ("DeepSpeed", "middle (config-driven)", "ZeRO 1/2/3 + offload", "very large models"),
    ("Accelerate", "top (thin wrapper)", "no sharding, only dispatch", "one codebase, many backends"),
]
widths = [16, 26, 30, 28]
for row in rows:
    line = " | ".join(f"{c:<{w}}" for c, w in zip(row, widths))
    print(line)
    if row[0] == "Method":
        print("-" * len(line))


Abstraction rises from bottom to top: flexibility falls, convenience rises. DDP / FSDP / DeepSpeed each implement one sharding strategy; Accelerate dispatches among them. Real projects often mix them — writing the main script in Accelerate, with FSDP or DeepSpeed as the backend.


## Summary

Confirm you understand the following:

- [ ] DDP holds a full model replica per card; throughput scales linearly but memory is not saved
- [ ] ZeRO shards redundancy across cards: Stage 1 shards optimizer state, Stage 2 adds gradients, Stage 3 adds parameters
- [ ] 7B model on 8 cards: DDP per-card 112 GB → ZeRO-3 per-card 14 GB
- [ ] Higher Stage saves more memory but costs more communication
- [ ] FSDP is PyTorch's built-in ZeRO-3; it tracks new hardware features fastest
- [ ] DeepSpeed drives Stage via JSON config and adds ZeRO-Offload to CPU/NVMe
- [ ] Accelerate is a top-level dispatcher; one codebase switches between DDP / FSDP / DeepSpeed
- [ ] The three solutions are not mutually exclusive; they can be combined (e.g., Accelerate + DeepSpeed)


## Exercises

> You may ask AI for hints on the approach, but it is not recommended to have AI "solve the exercise for you".

**Exercise 1: ZeRO Stage 3 per-card memory on 4 cards**

7B model, AdamW training, 4 cards. What is the per-card fixed memory under ZeRO Stage 3 (in GB)?

Hint: Stage 3 shards all 16P bytes across N cards, so per-card = 16P/N.


In [ ]:
# Exercise 1: ZeRO Stage 3 per-card memory on 4 cards
P = 7e9
N = 4

# TODO: compute Stage 3 per-card memory
s3_per_card_gb = None  # compute here

assert s3_per_card_gb is not None, "compute Stage 3 per-card memory first"
expected = 16 * P / N / 1e9
assert abs(s3_per_card_gb - expected) < 0.1, f"should be {expected:.1f} GB"
print(f"✅ Exercise 1 passed:")
print(f"   Stage 3 + 4 cards + 7B: per-card {s3_per_card_gb:.1f} GB")
print(f"   Compared to single-card DDP's 112 GB, compressed to {s3_per_card_gb / 112 * 100:.1f}%.")


**Exercise 2: DDP vs ZeRO-3 total memory**

7B model, 8 cards. What is the **total** memory (across all cards) under DDP and under ZeRO Stage 3? Why does ZeRO-3 seem to save a lot but the total only eliminates the redundant optimizer-state copies?

Hint: DDP has N full replicas, so total = N × 16P; ZeRO-3 has 16P/N per card, so total = 16P.


In [ ]:
# Exercise 2: DDP vs ZeRO-3 total memory
P = 7e9
N = 8

# TODO: compute DDP total memory and ZeRO-3 total memory (in GB)
ddp_total_gb = None
zero3_total_gb = None

assert ddp_total_gb is not None and zero3_total_gb is not None, "compute both totals first"
expected_ddp = N * 16 * P / 1e9
expected_zero3 = 16 * P / 1e9
assert abs(ddp_total_gb - expected_ddp) < 1, f"DDP total should be {expected_ddp:.0f} GB"
assert abs(zero3_total_gb - expected_zero3) < 1, f"ZeRO-3 total should be {expected_zero3:.0f} GB"
print(f"✅ Exercise 2 passed:")
print(f"   DDP total (8 cards):    {ddp_total_gb:.0f} GB")
print(f"   ZeRO-3 total (8 cards): {zero3_total_gb:.0f} GB")
print(f"   The saved {ddp_total_gb - zero3_total_gb:.0f} GB is all redundant copies.")


**Exercise 3: write a DeepSpeed config enabling ZeRO Stage 2**

Fill in the `deepspeed_config` dict below. Requirements: enable ZeRO Stage 2, do not offload to CPU, enable FP16.

Hint: see Section 5's example. Key fields are `zero_optimization.stage`, `zero_optimization.offload_optimizer.device`, `fp16.enabled`.


In [ ]:
# Exercise 3: DeepSpeed config for ZeRO Stage 2
import json

deepspeed_config = {
    # TODO: fill in the config here
    # Requirements:
    #   1. zero_optimization.stage == 2
    #   2. zero_optimization.offload_optimizer.device == "none"
    #   3. fp16.enabled == True
}

# Validation
assert "zero_optimization" in deepspeed_config, "missing zero_optimization field"
assert deepspeed_config["zero_optimization"]["stage"] == 2, "stage should be 2"
assert deepspeed_config["zero_optimization"]["offload_optimizer"]["device"] == "none", \
    "offload_optimizer.device should be 'none'"
assert deepspeed_config["fp16"]["enabled"] is True, "fp16.enabled should be True"

print("✅ Exercise 3 passed:")
print("Your DeepSpeed config:")
print(json.dumps(deepspeed_config, indent=2, ensure_ascii=False))
print()
print("This config enables ZeRO Stage 2, sharding gradients and optimizer state across cards while keeping parameters replicated.")


## References

- Rajbhandari et al., [ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054), 2020
- [PyTorch FSDP documentation](https://pytorch.org/docs/stable/fsdp.html)
- [DeepSpeed documentation](https://www.deepspeed.ai/)
- [HuggingFace Accelerate documentation](https://huggingface.co/docs/accelerate/)
- Zhao et al., [PyTorch FSDP](https://arxiv.org/abs/2304.11277), 2023
